In [ ]:
# Install dependencies.
%pip install anthropic python-dotenv

In [ ]:
#Load environment variables
from dotenv import load_dotenv
load_dotenv()

True

In [44]:
#Create client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-5-5"

In [ ]:
#Send
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

message.content[0].text

Quantum computing is a type of computation that uses quantum-mechanical phenomena such as superposition and entanglement to process information in ways that can solve certain problems much faster than classical computers.


In [47]:
#Conversation
def add_user_message(messages, content):
    messages.append({
        "role": "user",
        "content": content
    })

    return messages

def add_assistant_message(messages, content):
    messages.append({
        "role": "assistant",
        "content": content
    })

    return messages

def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    return client.messages.create(**params)

messages = []

response = chat(add_user_message(messages, "Define quantum computing in one sentence"))
add_assistant_message(messages, response.content[0].text)

response = chat(add_user_message(messages, "What is the topic?"))
add_assistant_message(messages, response.content[0].text)

messages[-1]["content"]

'The topic is quantum computing, specifically its definition: a form of computation that harnesses quantum-mechanical phenomena like superposition and entanglement to process information.'

In [ ]:
#Chat

messages = []

user_input = input("You: ")
while user_input:
    print("\n","You:", user_input)

    response = chat(add_user_message(messages, user_input))
    textBlock = next(x for x in response.content if x.type == "text")
    add_assistant_message(messages, textBlock.text)
    print("Assistant:", messages[-1]["content"])
    user_input = input("You: ")

print("Goodbye!")

In [ ]:
# system prompt

messages = []

system = """
    You are a senior python developer. You provide expert guidance and solutions in Python programming.
    You encourage the use of modern and concisee code.
"""

user_input = input("You: ")
while user_input:
    print("\n","You:", user_input)

    response = chat(add_user_message(messages, user_input), system)
    textBlock = next(x for x in response.content if x.type == "text")
    add_assistant_message(messages, textBlock.text)
    print("Assistant:", messages[-1]["content"])
    user_input = input("You: ")

print("Goodbye!")

In [48]:
# Structured data
messages = []

prompt = """
Generate three different sample AZ CLI commands. Each should be very short.
Return only the commands, one per line.
No introduction and no Markdown code fences.
"""

add_user_message(messages, prompt)
# add_assistant_message(messages, "Here are three short AZ CLI commands:")
text = chat(messages, stop_sequences=[])

text.content[0].text.strip()

'az group list --output table\naz vm list --output table\naz account show'

In [68]:
# Stream

messages = []

add_user_message(messages, "Generate 1 text about birds.")

with client.messages.stream(model=model, messages=messages, max_tokens=1000) as stream:
    for text in stream.text_stream:
        print(text, end="")

stream.get_final_message()["text"]

Birds are among the most diverse and widespread animals on Earth, with more than 10,000 known species living on every continent, including Antarctica. They have evolved remarkable adaptations for flight, such as lightweight hollow bones, feathers that provide both lift and insulation, and efficient respiratory systems that allow them to get oxygen even at high altitudes. Some birds, like the Arctic tern, migrate thousands of miles each year, while others, such as the bar-tailed godwit, can fly nonstop for over a week. Beyond their physical abilities, birds display a wide range of behaviors, from the intricate nest-building of weaver birds to the complex songs of songbirds, which are often used to attract mates and defend territories. Birds also play vital roles in ecosystems, pollinating flowers, dispersing seeds, and controlling insect populations, making them essential to the health of the planet.

TypeError: 'ParsedMessage' object is not subscriptable